# Prompt Engineering & AI Productivity Toolkit

**Google Colab practical document**

This notebook teaches prompt design patterns and builds reusable AI productivity workflows using Python. It runs without a paid API or API key by using illustrative prompt construction and deterministic examples. You can later connect an LLM provider in the optional extension section.

## Outcomes
- Design zero-shot and few-shot prompts.
- Structure prompts with role, context, instructions, examples, and constraints.
- Request JSON and validate its schema.
- Build summarization, classification, and generation workflows.
- Practice AI-assisted coding/debugging safely.
- Evaluate prompt quality with reusable tests.
- Assemble a small productivity toolkit.

## 1. Setup
Run this cell first. The notebook uses only Python's standard library and pandas for the evaluation report.

In [ ]:
import json
import re
import textwrap
import pandas as pd
from typing import Any

print("Environment ready")

## 2. Prompt engineering fundamentals

A useful prompt commonly contains:
- **Role:** the perspective or expertise to adopt.
- **Context:** background and source material.
- **Instruction:** the task to perform.
- **Constraints:** boundaries such as length, tone, audience, and exclusions.
- **Output format:** the structure expected in the response.

A prompt should be specific enough to reduce ambiguity, but not so restrictive that it blocks useful answers.

## 3. Zero-shot prompting

Zero-shot prompting asks the model to perform a task without examples.

In [ ]:
zero_shot_prompt = '''
Role: You are a customer-support analyst.
Context: A customer says, "The delivery arrived two days late, but the package is undamaged."
Task: Classify the issue.
Allowed labels: Delivery Delay, Damaged Item, Missing Item, Other.
Constraints: Choose exactly one label and give one short reason.
Output: JSON with keys "label" and "reason".
'''
print(zero_shot_prompt)

## 4. Few-shot prompting

Few-shot prompting supplies examples that demonstrate the desired pattern. The examples should be representative and should not contradict one another.

In [ ]:
few_shot_prompt = '''
Role: You classify customer feedback.
Instructions: Select one label: Positive, Negative, or Neutral.

Examples:
Feedback: "The product works perfectly. I am happy."
Label: Positive

Feedback: "It stopped working after one day."
Label: Negative

Feedback: "The parcel arrived on Tuesday."
Label: Neutral

Now classify:
Feedback: "The delivery was late, but the product is excellent."
Label:
'''
print(few_shot_prompt)

## 5. Prompt components: role, context, instructions, constraints

Use a reusable function to assemble prompts consistently.

In [ ]:
def build_prompt(role: str, context: str, instruction: str,
                 constraints: list[str], output_format: str) -> str:
    constraint_text = "\\n".join(f"- {item}" for item in constraints)
    return f'''ROLE:\\n{role}

CONTEXT:\\n{context}

INSTRUCTION:\\n{instruction}

CONSTRAINTS:\\n{constraint_text}

OUTPUT FORMAT:\\n{output_format}
'''

example_prompt = build_prompt(
    role="You are an experienced technical writer.",
    context="The audience is a group of first-year engineering students.",
    instruction="Explain what an API is.",
    constraints=["Use simple English", "Use one practical example", "Stay under 150 words"],
    output_format="A short explanation with a heading and one example."
)
print(example_prompt)

## 6. Structured output: JSON

Ask for a precise schema and validate the result. JSON parsing catches syntax errors; additional checks validate the meaning and allowed values.

In [ ]:
sample_json = '''
{
  "category": "Delivery Delay",
  "priority": "Medium",
  "summary": "The delivery arrived two days late and the package is undamaged."
}
'''

record = json.loads(sample_json)
required_keys = {"category", "priority", "summary"}
allowed_priorities = {"Low", "Medium", "High"}

assert required_keys.issubset(record.keys())
assert record["priority"] in allowed_priorities
print("Valid JSON and required fields:", record)

## 7. Summarization workflow

The example below extracts a short summary from a supplied paragraph. In a real LLM workflow, the prompt would be sent to the model; always compare the result with the original source.

In [ ]:
source_text = '''
The engineering team completed the first prototype of the monitoring dashboard.
The dashboard currently displays equipment status and temperature readings.
The team will add alert history next week. The operations manager requested a
short user guide before the pilot begins. No launch date has been approved.
'''

def simple_summary(text: str, max_sentences: int = 2) -> str:
    sentences = re.split(r'(?<=[.!?])\\s+', text.strip())
    return " ".join(sentences[:max_sentences])

summary_prompt = build_prompt(
    role="You are a concise project analyst.",
    context=source_text,
    instruction="Summarize the source for a project manager.",
    constraints=["Do not add facts", "Mention pending work", "Use no more than two sentences"],
    output_format="Plain text"
)
print("Prompt:\\n", summary_prompt)
print("Demonstration summary:\\n", simple_summary(source_text))

## 8. Classification workflow

Classification maps input text to a fixed set of labels. The demonstration uses transparent keyword rules; an LLM-based classifier should be evaluated against labeled examples.

In [ ]:
ALLOWED_LABELS = {"Positive", "Negative", "Neutral"}

def classify_feedback(text: str) -> str:
    lowered = text.lower()
    positive_words = ["excellent", "great", "happy", "perfect"]
    negative_words = ["broken", "late", "poor", "failed", "damaged"]
    pos = any(word in lowered for word in positive_words)
    neg = any(word in lowered for word in negative_words)
    if pos and not neg:
        return "Positive"
    if neg and not pos:
        return "Negative"
    return "Neutral"

examples = [
    "The product is excellent.",
    "The item arrived damaged.",
    "The package arrived on Monday."
]
for item in examples:
    label = classify_feedback(item)
    assert label in ALLOWED_LABELS
    print(item, "=>", label)

## 9. Generation workflow

Generation prompts should specify audience, purpose, tone, length, and any facts that must be preserved. The following function creates a prompt, not an AI-generated answer.

In [ ]:
generation_prompt = build_prompt(
    role="You are a professional learning-content writer.",
    context="A beginner is learning Python for data analysis.",
    instruction="Create a short LinkedIn learning update about completing a first pandas exercise.",
    constraints=[
        "Use a professional and authentic tone",
        "Do not claim employment or certification",
        "Keep it under 80 words"
    ],
    output_format="One short paragraph and up to three relevant hashtags."
)
print(generation_prompt)

## 10. AI-assisted coding and debugging

A reliable coding prompt includes:
1. Expected behavior.
2. Actual behavior and full error message.
3. Relevant code and input.
4. Runtime/library versions.
5. Constraints and a request for tests.

Do not paste secrets, credentials, private customer data, or proprietary code into an unapproved service. Review generated code and run tests before using it.

In [ ]:
buggy_code = '''
def average(values):
    return sum(values) / len(values)

print(average([]))
'''

debug_prompt = build_prompt(
    role="You are a senior Python developer.",
    context=buggy_code,
    instruction="Identify the failure and propose a safe correction.",
    constraints=[
        "Explain the root cause",
        "Handle an empty list explicitly",
        "Include a test for empty and non-empty input"
    ],
    output_format="Root cause, corrected code, and tests"
)
print(debug_prompt)

In [ ]:
def safe_average(values):
    if not values:
        return None
    return sum(values) / len(values)

assert safe_average([2, 4, 6]) == 4
assert safe_average([]) is None
print("Debugging exercise tests passed")

## 11. Prompt evaluation framework

Evaluate prompts on multiple dimensions:
- **Task success:** does the answer complete the requested task?
- **Groundedness:** are claims supported by the provided context?
- **Format validity:** does the answer match the requested schema?
- **Consistency:** does it behave similarly across equivalent inputs?
- **Safety/privacy:** does it avoid unsafe or unnecessary sensitive content?

Use a fixed test set, record failures, revise one prompt element at a time, and rerun the same tests.

In [ ]:
test_cases = [
    {"test_id": "JSON-01", "dimension": "Format validity", "passed": True,
     "notes": "JSON parses and required keys exist."},
    {"test_id": "CLS-01", "dimension": "Task success", "passed": True,
     "notes": "Predicted label belongs to allowed set."},
    {"test_id": "SUM-01", "dimension": "Groundedness", "passed": True,
     "notes": "Summary uses only supplied source statements."},
    {"test_id": "COD-01", "dimension": "Code verification", "passed": True,
     "notes": "Corrected function passes included assertions."},
]
evaluation_df = pd.DataFrame(test_cases)
evaluation_df

In [ ]:
pass_rate = evaluation_df["passed"].mean() * 100
print(f"Test pass rate: {pass_rate:.1f}%")
evaluation_df.to_csv("prompt_evaluation_report.csv", index=False)
print("Saved prompt_evaluation_report.csv")

## 12. Build the AI Productivity Toolkit

Reusable workflow categories:
- **Summarization:** meeting notes, long documents, action items.
- **Email:** draft, rewrite, adjust tone, extract action items.
- **Resume:** improve clarity and impact without inventing metrics.
- **Coding:** explain code, debug errors, generate tests, review changes.
- **Research:** define a question, synthesize supplied sources, list evidence gaps.

Use the templates in `templates/prompt_templates.json` as starting points. Replace placeholders with task-specific context and verify outputs before relying on them.

In [ ]:
toolkit = {
    "summarization": "Summarize [SOURCE] for [AUDIENCE]. Return key points, decisions, and action items. Do not add facts.",
    "email": "Draft an email to [RECIPIENT] about [PURPOSE]. Tone: [TONE]. Include only these facts: [FACTS].",
    "resume": "Improve this resume bullet: [BULLET]. Preserve facts; do not invent metrics. Target role: [ROLE].",
    "coding": "Review this code: [CODE]. Expected behavior: [EXPECTED]. Actual behavior: [ACTUAL]. Explain root cause and provide tests.",
    "research": "Research question: [QUESTION]. Use only these sources: [SOURCES]. Separate findings, interpretations, and evidence gaps."
}
for name, template in toolkit.items():
    print(f"\\n{name.upper()}\\n{template}")

## 13. Prompt testing checklist

Before using a prompt in a real workflow:
- [ ] The task and audience are explicit.
- [ ] Context is relevant and sufficient.
- [ ] Constraints are measurable where possible.
- [ ] Output format is defined.
- [ ] Examples are representative and consistent.
- [ ] Edge cases are included in the test set.
- [ ] Outputs are checked for factuality, bias, privacy, and safety.
- [ ] Prompt version and test results are recorded.

## 14. Optional extension: connect an LLM

To turn this prompt toolkit into a live application, choose an approved model provider and install its official SDK. Store the API key in **Google Colab Secrets** or an environment variable; never place it directly in notebook cells or GitHub.

A provider integration should:
1. Read the secret securely.
2. Send the assembled prompt to the model.
3. Parse and validate structured outputs.
4. Handle rate limits, timeouts, and malformed responses.
5. Log prompt version and evaluation results without storing sensitive content unnecessarily.

## Final deliverables
- GitHub-ready repository structure and reusable prompt templates.
- This executable Google Colab notebook.
- A sample prompt evaluation report.